# Plan 005.1 — Targeted Catalog Updates and Snapshot Retention

**Audience:** Python users who know basic files and Catalog scans.  
**Prerequisites:** Python 3.12+ and the repository environment (`uv sync`). Launch
with `uv run jupyter lab notebooks/public` and select the project Python kernel.

By the end, you can register new files, update existing observations, retire
catalog paths, retain two successful snapshots, and perform a full refresh.

1. Create temporary files and a baseline catalog.
2. Register additions without scanning unrelated files.
3. Update a file and inspect two-snapshot retention.
4. Remove a catalog instance without deleting its physical file.
5. Refresh the entire directory and reopen the database.
6. Practice keep-all retention and clean up.

All file creation, editing, and cleanup stays inside `TemporaryDirectory`.
Catalog methods only observe files and write the temporary database. These are
Catalog features, not an Archiver copy/delete demonstration; Plan 005.2 is outside
this tutorial. Run all cells in order. Restarting and running all cells creates a
fresh sandbox.

References: [Plan 005.1](../../docs/plans/005.1-targeted-catalog-updates.md),
[Catalog API](../../docs/catalog-api.md#targeted-inventory-updates), and
[domain model](../../docs/domain-model.md).

## 1. Create a temporary catalog and baseline

The database lives beside the source directory, so scans do not encounter it.
Three files form the baseline. Later we will add a duplicate and an empty file.

In [1]:
from hashlib import sha256
from pathlib import Path, PurePosixPath
from tempfile import TemporaryDirectory

from archiver import Catalog

# Re-running setup also releases this tutorial's previous temporary resources.
if "catalog" in globals():
    catalog.close()
if "temporary" in globals():
    temporary.cleanup()

temporary = TemporaryDirectory(prefix="archiver-plan-0051-")
workspace = Path(temporary.name)
root = workspace / "source"
root.mkdir()
database = workspace / "catalog.sqlite"
(root / "keep.txt").write_bytes(b"shared content")
(root / "note.txt").write_bytes(b"original note")
(root / "retired.txt").write_bytes(b"retire this instance")
catalog = Catalog.create(database)
assert catalog.schema_version == 3
print("Temporary source and schema-3 Catalog created.")

Temporary source and schema-3 Catalog created.


These helpers show known file identities and snapshot history. They consume each
query fully, leaving no active streams when later operations prune old snapshots.
The filesystem snapshot lets us verify that Catalog operations preserve bytes,
mtime, and mode. The tutorial itself edits files only when explicitly shown.

In [2]:
def inventory():
    return {
        item.relative_path.as_posix(): item.content_id.digest
        for item in catalog.files(root).stream_all()
    }


def source_state():
    return {
        path.relative_to(root).as_posix(): (
            path.read_bytes(), path.stat().st_mtime_ns, path.stat().st_mode
        )
        for path in root.rglob("*") if path.is_file()
    }


def history():
    return list(catalog.scan_summaries(root))


def show_state():
    print("Known files (path | SHA-256 prefix):")
    for relative, digest in sorted(inventory().items()):
        print(f"  {relative} | {digest[:12]}")
    print("Successful snapshots, newest first (ID | origin | known files):")
    for item in history():
        print(f"  {item.scan.id} | {item.scan.origin} | {item.files_observed}")


before = source_state()
catalog.scan_directory(root)
assert source_state() == before
assert set(inventory()) == {"keep.txt", "note.txt", "retired.txt"}
assert len(history()) == 1
baseline_scan = catalog.current_scan(root)
show_state()

Known files (path | SHA-256 prefix):
  keep.txt | 150f3319880a
  note.txt | e5a47819e9d3
  retired.txt | 9ea4084e4643
Successful snapshots, newest first (ID | origin | known files):
  1 | full_scan | 3


## 2. Register additions without a full scan

Create two files: one duplicates `keep.txt`; one is empty. `prepare_file_update`
observes only the supplied paths and writes no inventory. Applying those changes
publishes a complete successor inventory, inheriting the three untouched records.
Relative input paths are relative to `root`, not the notebook's working directory.

In [ ]:
(root / "incoming").mkdir()
(root / "incoming/copy.txt").write_bytes(b"shared content")
(root / "incoming/empty.bin").write_bytes(b"")
known_before = inventory()
before = source_state()

changes = catalog.prepare_file_update(
    root, paths=[Path("incoming/copy.txt"), Path("incoming/empty.bin")]
)
assert inventory() == known_before  # Preparation has not published anything.
added = catalog.apply_inventory_changes(changes)  # keep_last=2 is the default.

assert (added.inserted_files, added.replaced_files, added.removed_files) == (2, 0, 0)
assert inventory()["incoming/copy.txt"] == inventory()["keep.txt"]
assert inventory()["incoming/empty.bin"] == sha256(b"").hexdigest()
assert catalog.duplicates(root).summary().duplicate_content_group_count == 1
assert len(history()) == 2
assert source_state() == before
show_state()

## 3. Update an existing file and keep two successful snapshots

Edit `note.txt` in the temporary source, then register only that file. Forced
hash verification makes the demonstration independent of metadata-cache reuse.
Normally matching path, size, and mtime can reuse a cached identity.

The third successful publication keeps snapshots 2 and 3 and prunes snapshot 1.
This retention and the new current inventory commit together. A failed database
publication would preserve the previous committed inventory and history.

In [ ]:
old_digest = inventory()["note.txt"]
(root / "note.txt").write_bytes(b"a revised note with different bytes")
before = source_state()
changes = catalog.prepare_file_update(root, paths=Path("note.txt"), verify_hashes=True)
updated = catalog.apply_inventory_changes(changes)

assert (updated.inserted_files, updated.replaced_files, updated.removed_files) == (0, 1, 0)
assert inventory()["note.txt"] != old_digest
assert inventory()["note.txt"] == sha256((root / "note.txt").read_bytes()).hexdigest()
assert len(history()) == 2
assert baseline_scan is not None
assert baseline_scan.id not in [item.scan.id for item in history()]
assert updated.retention is not None and updated.retention.scans_to_remove == 1
assert source_state() == before
print("Replaced one observation; pruned one old successful snapshot.")
show_state()

## 4. Retire a catalog path without deleting the file

A removal changes the inventory only. It does not unlink, move, or tag the file.
`retired.txt` deliberately stays on disk so the full refresh can discover it again.
Content identities remain known even when they have no current file instance;
content-level tags, if present, also survive snapshot pruning.

In [ ]:
retired_identity = catalog.content_for_path(root, PurePosixPath("retired.txt"))
before = source_state()
changes = catalog.prepare_file_update(root, remove_paths=[PurePosixPath("retired.txt")])
removed = catalog.apply_inventory_changes(changes)

assert (removed.inserted_files, removed.replaced_files, removed.removed_files) == (0, 0, 1)
assert "retired.txt" not in inventory()
assert (root / "retired.txt").read_bytes() == b"retire this instance"
assert any(row.content_id == retired_identity for row in catalog.contents().fetch())
assert len(history()) == 2
assert source_state() == before
print("Removed from current inventory; physical file and content identity retained.")
show_state()

## 5. Perform a full refresh afterward

Create one more file without registering it. The current inventory cannot know
about this new path yet. A full refresh will find both this file and the retired
file still on disk.

Full scans retain all history by default for compatibility. Pass `keep_last=2`
explicitly to maintain the same two-snapshot policy. Both full scans and targeted
updates count toward that limit. Snapshot origin describes how an inventory was
published; success is not a guarantee that the filesystem will stay unchanged.

In [ ]:
(root / "unregistered.txt").write_bytes(b"discovered by the full refresh")
assert "unregistered.txt" not in inventory()
before = source_state()
refresh = catalog.reconcile_directory(root, verify_hashes=True)
new_paths = {
    change.relative_path.as_posix()
    for change in refresh.changes if change.kind == "new"
}
assert new_paths == {"retired.txt", "unregistered.txt"}
print("Full refresh will discover:", sorted(new_paths))

# scan_directory performs reconciliation and publication as one convenient call.
summary = catalog.scan_directory(root, verify_hashes=True, keep_last=2)
assert set(inventory()) == {
    "keep.txt", "note.txt", "retired.txt", "incoming/copy.txt",
    "incoming/empty.bin", "unregistered.txt",
}
assert summary.files_observed == 6
assert len(history()) == 2
assert [item.scan.origin for item in history()] == ["full_scan", "targeted_update"]
assert source_state() == before
show_state()

Reopening the database preserves the refreshed inventory and its two retained
snapshots. This is a new schema-3 catalog: opening schema 1 or 2 instead requires
an explicit rebuild in a separate database. Rescanning recovers file inventory,
but does not reconstruct old user tags or provenance.

In [ ]:
expected_inventory = inventory()
expected_history = [(item.scan.id, item.scan.origin) for item in history()]
catalog.close()
catalog = Catalog.open(database)
assert inventory() == expected_inventory
assert [(item.scan.id, item.scan.origin) for item in history()] == expected_history
print("Reopen verified: six known files and two successful snapshots.")

## 6. Exercise: retain all snapshots for one publication

Before running the answer below, predict what happens if you register unchanged
`keep.txt` with `keep_last=None`. How many snapshots remain? Does the inventory
change? Does the revision advance?

Optional extension: change that policy to `keep_last=1`, restart the kernel, and
adapt the exercise assertions to check that only the new snapshot survives.

In [ ]:
# Answer scaffold: None explicitly disables pruning for this publication.
old_inventory = inventory()
old_count = len(history())
unchanged = catalog.prepare_file_update(root, paths=Path("keep.txt"))
answer = catalog.apply_inventory_changes(unchanged, keep_last=None)
assert inventory() == old_inventory
assert answer.replaced_files == 1
assert answer.revision != unchanged.baseline
assert len(history()) == old_count + 1 == 3
assert answer.retention is None
print("Unchanged nonempty update: new revision, unchanged inventory, three retained snapshots.")

# Return to the tutorial's two-snapshot policy.
catalog.apply_inventory_changes(catalog.prepare_file_update(root, paths=Path("keep.txt")))
assert len(history()) == 2
print("Default policy restored: two successful snapshots retained.")

## Common pitfalls and boundaries

- Retiring a path from inventory does not delete its file. Refresh finds it again
  while it remains on disk.
- Registering selected files carries forward prior knowledge about untouched
  paths. It does not perform a full scan or guarantee directory completeness.
- An empty request publishes nothing; an unchanged nonempty request publishes a
  new snapshot. Keep-all avoids pruning, not snapshot creation.
- Close active streams before requesting pruning. Handles referring to pruned
  history cannot read those deleted snapshots afterward.
- SQLite publication and physical file changes cannot share an atomic transaction.
  This tutorial prepares its temporary files before observing them.
- The current implementation copies inventory rows within SQLite per batch.
  Targeted observation avoids unrelated filesystem reads; database writes are
  still snapshot-based. No Plan 005.2 comparison or Archiver execution is used.

## 7. Release temporary resources

Close the Catalog before cleaning up its temporary directory, especially on
Windows. You can run the entire notebook again to create another fresh sandbox.

In [ ]:
catalog.close()
temporary.cleanup()
assert not workspace.exists()
print("Verified all examples; temporary source files and database cleaned up.")